In [1]:
# open json file and print the content
import json


def update_launch_json(experiment, version):

    with open(".vscode/launch.json") as f:
        data = json.load(f)

    new_launch = data["configurations"][0].copy()
    name = f"{experiment}_{version}"
    new_launch["name"] = name
    new_launch["preLaunchTask"] = name
    new_launch["program"] = (
        "${workspaceFolder}" + f"/bazel-bin/experiments/{experiment}/{version}/exp"
    )

    launch_skip = False
    for i in data["configurations"]:
        if i["name"] == name:
            print(f"launch {name} already exists")
            launch_skip = True
            break

    if not launch_skip:
        data["configurations"].append(new_launch)
        save_path = ".vscode/launch.json"
        with open(save_path, "w") as f:
            json.dump(data, f, indent=2)

    with open(".vscode/tasks.json") as f:
        tasks = json.load(f)
    new_task = tasks["tasks"][0].copy()
    new_task["label"] = name
    command = f"bazel6 build //experiments/{experiment}/{version}:exp -c dbg --cxxopt='-DSYSC' --cxxopt='-DACC_PROFILE' --spawn_strategy=standalone  --subcommands  --platforms=//platform:linux_x64 --@secda_tools//:config=sysc"
    new_task["command"] = command

    task_skip = False
    for i in tasks["tasks"]:
        if i["label"] == name:
            print(f"task {name} already exists")
            task_skip = True
            break

    if not task_skip:
        tasks["tasks"].append(new_task)
        save_path = ".vscode/tasks.json"
        with open(save_path, "w") as f:
            json.dump(tasks, f, indent=2)
    

update_launch_json("dma_exp", "v4")
